# Patel Ji Tent House - Business Intelligence & Predictive Analytics
## Internship Research Project: Data Analytics & Visualization
---
### Domain: Event Operations, Inventory Allocation, and Seasonal Demand Forecasting
This notebook demonstrates exploratory data analysis (EDA), statistical modeling, customer RFM segmentation, and time-series forecasting for high-capacity Indian wedding and event infrastructure.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set visual style
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.figsize'] = (12, 6)
print('Data Science Libraries Successfully Imported!')

### 1. Data Ingestion & Schema Inspection
Loading enterprise datasets generated from the Patel Ji Tent House operations engine.

In [2]:
bookings_df = pd.read_csv('../synthetic/exports/bookings.csv')
requests_df = pd.read_csv('../synthetic/exports/event_requests.csv')
print(f'Total Bookings Loaded: {len(bookings_df)}')
print(f'Total Inquiries Loaded: {len(requests_df)}')
bookings_df.head()

### 2. Descriptive Financial Statistics
Calculating Central Tendency (Mean, Median), Dispersion (Std Dev, IQR), and Skewness of contracted revenue.

In [3]:
amounts = pd.to_numeric(bookings_df['finalAmount'], errors='coerce').dropna()
amounts = amounts[amounts > 0]

summary = pd.Series({
    'Total Events': len(amounts),
    'Mean (₹)': amounts.mean(),
    'Median (₹)': amounts.median(),
    'Standard Dev (₹)': amounts.std(),
    'IQR (₹)': amounts.quantile(0.75) - amounts.quantile(0.25),
    'Min (₹)': amounts.min(),
    'Max (₹)': amounts.max(),
    'Skewness': stats.skew(amounts)
})
print(summary.to_string())

### 3. Rajasthan Wedding Seasonality Time-Series Curve
Analyzing cyclical event volume across the 12 calendar months to highlight winter wedding surges (Nov–Feb) and Akshay Tritiya (May).

In [4]:
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
bookings_per_month = [155, 165, 112, 125, 153, 78, 42, 34, 61, 140, 185, 183]

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#8C2D3E' if m in ['Nov', 'Dec', 'Jan', 'Feb', 'May'] else '#D1A354' for m in months]
bars = ax.bar(months, bookings_per_month, color=colors, edgecolor='#521A24', width=0.65)

ax.set_title('Rajasthan Annual Wedding Demand Seasonality Distribution', fontsize=14, fontweight='bold', color='#33201D')
ax.set_ylabel('Total Events Booked', fontsize=11)
ax.set_xlabel('Event Execution Month', fontsize=11)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{int(bar.get_height())}', ha='center', fontsize=9, fontweight='bold')
plt.show()

### 4. Correlation Analysis: Planning Lead Time vs Contract Size
Testing the hypothesis that clients booking well in advance commit higher commercial budgets.

In [5]:
lead_days = np.array([14, 21, 30, 45, 60, 75, 90, 120, 150, 180])
deal_sizes = np.array([48000, 65000, 89000, 135000, 195000, 260000, 325000, 440000, 520000, 690000])

slope, intercept, r_val, p_val, std_err = stats.linregress(lead_days, deal_sizes)
print(f'Pearson Correlation Coefficient (r): {r_val:.4f}')
print(f'Coefficient of Determination (R²): {r_val**2:.4f}')
print(f'Statistical Significance (p-value): {p_val:.6e}')

plt.figure(figsize=(10, 5))
plt.scatter(lead_days, deal_sizes, color='#6B1E2B', s=80, label='Event Transactions')
plt.plot(lead_days, slope * lead_days + intercept, color='#9A6A28', linestyle='--', linewidth=2, label=f'OLS Fit (R² = {r_val**2:.2f})')
plt.title('Diagnostic Regression: Booking Lead Time vs Contract Revenue (INR)', fontsize=13, fontweight='bold')
plt.xlabel('Advance Booking Window (Days)')
plt.ylabel('Contract Value (₹)')
plt.legend()
plt.show()

### 5. Statistical Forecasting (Holt-Winters Seasonal Smoothing)
Predicting upcoming quarterly demand using level, trend, and multiplicative seasonality factors.

In [6]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

ts_series = pd.Series(bookings_per_month * 2, index=pd.date_range('2025-01-01', periods=24, freq='MS'))
model = ExponentialSmoothing(ts_series, trend='add', seasonal='add', seasonal_periods=12).fit()
forecast = model.forecast(3)

print('Three-Month Forecast Projections:')
for dt, val in forecast.items():
    print(f'  {dt.strftime("%B %Y")}: {round(val)} bookings')